use only if you have downloaded corrupt cloned of shap-e


In [ ]:
#!rm -rf shap-e #(use only when there is error during cloning) undone the #


In [ ]:
!git clone https://github.com/openai/shap-e.git
%cd shap-e
!pip install -e .
#!pip install -r requirements.txt  #there is no such file in shap-e folder but there is setup.py file


In [ ]:
!pip install trimesh ipywidgets


Here we are importing libraries for our projject


In [ ]:

import torch
import ipywidgets as widgets
import trimesh
from shap_e.diffusion.sample import sample_latents
from shap_e.diffusion.gaussian_diffusion import diffusion_from_config
from shap_e.models.download import load_model, load_config
from shap_e.util.notebooks import decode_latent_mesh
from IPython.display import display, FileLink


Loading models and setting up device

In [ ]:
# setting   up Gpu or CpU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load pre-trained models
final_builder_model = load_model('transmitter', device=device)
model = load_model('text300M', device=device)
diffusion = diffusion_from_config(load_config('diffusion'))


text_input = widgets.Text(
    description="Enter Prompt:",
    placeholder="Rahul Kumar"
)

output = widgets.Output()

# button for triggering model generation
generate_button = widgets.Button(description="Generate 3D Model")


In [ ]:
import trimesh

def convert_3D_mesh_to_stl(mesh, stl_path):
    # Convert OpenAI TriMesh to trimesh-compatible format
    vertices = mesh.verts
    faces = mesh.faces

    tri_mesh = trimesh.Trimesh(vertices=vertices, faces=faces, process=False)
    tri_mesh.export(stl_path)


In [ ]:
from google.colab import files


def generate_3d_model(change):
    output.clear_output()  # use it to clear past images

    prompt = text_input.value.strip()

    # If no prompt is entered
    if not prompt:
        with output:
            print("❗⚠️ Oops! You forgot to enter a description. Try something like 'a futuristic chair' or 'a dragon statue'..")
        return

    with output:
        print(f"🚀 wait while wi generating model: '{prompt}'")

    # Generate the latent vector
    latents = sample_latents(
        batch_size=1,
        model=model,
        diffusion=diffusion,
        guidance_scale=15.0,
        model_kwargs=dict(texts=[prompt]),
        progress=True,
        clip_denoised=True,
        use_fp16=torch.cuda.is_available(),
        use_karras=True,
        karras_steps=64,
        sigma_min=1e-3,
        sigma_max=160,
        s_churn=0,
    )

    # Decode to mesh
    mesh = decode_latent_mesh(final_builder_model, latents[0]).tri_mesh()

    # File paths
    obj_path = "/content/output_model.obj"
    stl_path = "/content/output_model.stl"

    # Save OBJ
    with open(obj_path, 'w') as f:
        mesh.write_obj(f)

    # Save STL using trimesh
    convert_3D_mesh_to_stl(mesh, stl_path)

    with output:
        print("✅ finally Model generated!")
        print("⬇️ Click the link  to download:")
        files.download(obj_path)
        files.download(stl_path)

    # Optionally display 3D model (optional and may fail if pyglet or OpenGL is missing)
    try:
        display_3d_model(obj_path)
    except Exception as e:
        with output:
            print("⚠️ Failed to display the 3D model:", str(e))


In [ ]:
import trimesh
from trimesh.viewer import scene_to_notebook
from IPython.display import display

# This function will display the 3D model in Colab using trimesh
def display_3d_model(obj_path):
    with output:
        print("Displaying 3D model using trimesh...")

        # Load the 3D mesh from the .obj file
        mesh = trimesh.load_mesh(obj_path)

        # Display the 3D mesh
        try:
            scene = mesh.scene()  # Create a scene from the mesh
            display(scene_to_notebook(scene))  # Render the scene inline in the notebook-----if you are using another platform beside google collab once do chatgpt.
        except Exception as e:
            print(f"Failed to display the 3D model: {e}")


In [ ]:
# Link the button to the model generation function
generate_button.on_click(generate_3d_model)


display(text_input, generate_button, output)
